<div style="background:#1A2E4A; color:white; padding:32px 40px; border-radius:10px; margin-bottom:8px">
  <p style="color:#14A085; font-size:13px; margin:0 0 6px 0; letter-spacing:1px">TRATTAMENTO DI DATI MULTIMEDIALI — Lezioni Pratiche · Blocco A</p>
  <h1 style="margin:0 0 8px 0; font-size:32px">Lezione Pratica 1</h1>
  <h2 style="margin:0; font-weight:300; font-size:22px; color:#94D1C8">Manipolazione di immagini con Python (Pillow)</h2>
  <hr style="border-color:#0D7377; margin:20px 0">
  <p style="margin:0; font-size:13px; color:#94A3B8">📚 Lezioni teoriche di riferimento: 2 (Rappresentazione digitale), 3 (Formati popolari)</p>
  <p style="margin:4px 0 0 0; font-size:13px; color:#94A3B8">🛠 Strumenti: Python · Pillow · matplotlib · NumPy</p>
</div>

## Prima di iniziare

Prerequisiti: variabili, liste, cicli e chiamate di funzione in Python. Gli array NumPy vengono introdotti durante la lezione.

Una cella **Markdown** contiene testo; una cella **Code** esegue Python. Usate **Shift+Invio** per eseguire la cella e passare alla successiva. Le variabili rimangono in memoria: dopo modifiche al setup, riavviate il kernel ed eseguite tutte le celle dall'alto.

I file generati vengono salvati in `output`, accanto al notebook. La fotografia è inclusa nella cartella `data`: non occorre scaricare immagini durante la lezione. In Colab caricate anche la cartella `data` con la fotografia.


## Obiettivi della lezione

Al termine di questa sessione sarete in grado di:

1. **Caricare e ispezionare** immagini raster con Pillow, leggendone attributi fondamentali
2. **Applicare trasformazioni geometriche** (ridimensionamento, ritaglio, rotazione, flip)
3. **Convertire tra modalità** (RGB, scala di grigio, RGBA, palette)
4. **Salvare in formati diversi** (BMP, PNG, JPEG) e misurarne la dimensione su disco
5. **Visualizzare e confrontare** i risultati, osservando gli artefatti introdotti dalla compressione JPEG

---

### Struttura del notebook

| Sezione | Contenuto | Tempo stimato |
|---------|-----------|---------------|
| **0** | Uso del notebook, setup e dataset | 10 min |
| **1** | Caricare, ispezionare e leggere i pixel | 15 min |
| **2** | Operazioni geometriche e proporzioni | 20 min |
| **3** | RGB, grigio e trasparenza | 15 min |
| **4–5** | Formati, dimensioni e confronto visivo JPEG | 20 min |
| **6** | Un esercizio guidato e discussione | 10 min |

> 💡 **Come usare questo notebook:** eseguite le celle nell'ordine. Le celle marcate con 🔧 contengono esercizi da completare voi. Le celle marcate con ✅ sono già complete e producono output di riferimento.

**Percorso di 90 minuti:** gli esercizi delle sezioni 1–4 si svolgono insieme al docente nei tempi indicati. Palette, CMYK, misure PSNR e gli esperimenti JPEG avanzati sono approfondimenti facoltativi. Nella discussione finale scegliete un solo esperimento, per esempio il confronto PNG/JPEG sul gradiente.


---
## Sezione 0 — Setup e dataset

Prepariamo le immagini di test. Usiamo **due immagini**:
una sintetica (gradiente colorato), per avere un comportamento prevedibile, e una fotografica (una formica), per avvicinarci all'uso reale. La fotografia si trova in `data/photo.jpg`.

La fotografia originale è rettangolare (960×640): per ottenere un'immagine 512×512 senza deformarla la ridimensioniamo a 512×341 e aggiungiamo **bande bianche** sopra e sotto (`ImageOps.pad`). Le bande uniformi incidono sulle misure di compressione delle sezioni 4–5: tenetelo presente quando interpretate le dimensioni dei file.

La sorgente è JPEG: il riferimento PNG preserva i pixel decodificati e preparati per questo esperimento, non recupera dettagli già persi nel JPEG originale.

In [ ]:
# Installazione dipendenze (eseguire solo se necessario)
# %pip install pillow matplotlib numpy --quiet

In [ ]:
# ✅ Import e configurazione globale
import os
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

# Cartella di lavoro per i file generati
# Supporta avvio dalla cartella 001 oppure dalla radice del repository.
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "data" / "photo.jpg").is_file():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "001"
PHOTO_PATH = NOTEBOOK_DIR / "data" / "photo.jpg"
if not PHOTO_PATH.is_file():
    raise FileNotFoundError("Manca data/photo.jpg: aprite il notebook dalla cartella 001 o dalla radice del repository e includete la cartella data.")
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# Stile grafici
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
})

print("✅ Import completati")

In [ ]:
# ✅ Generazione del dataset di test

# ── 1. Immagine sintetica: gradiente 2D RGB ──────────────────────────────────
def make_gradient(width=512, height=512):
    """Gradiente: R varia sull'asse X, G sull'asse Y, B è la media."""
    r = np.linspace(0, 255, width, dtype=np.uint8)[np.newaxis, :].repeat(height, axis=0)
    g = np.linspace(0, 255, height, dtype=np.uint8)[:, np.newaxis].repeat(width, axis=1)
    b = ((r.astype(np.float32) + g.astype(np.float32)) / 2).astype(np.uint8)
    return Image.fromarray(np.stack([r, g, b], axis=2))

img_synth = make_gradient()
img_synth.save(OUTPUT_DIR / "synth.png")
print(f"Immagine sintetica : {img_synth.size}  mode={img_synth.mode}")

# ── 2. Fotografia locale: manteniamo le proporzioni ─────────────────────────
with Image.open(PHOTO_PATH) as source:
    photo_rect = ImageOps.exif_transpose(source).convert("RGB")
img_photo = ImageOps.pad(photo_rect, (512, 512),
                         method=Image.Resampling.LANCZOS, color="white")
img_photo.save(OUTPUT_DIR / "photo.png")
print(f"Immagine fotografica: {img_photo.size}  mode={img_photo.mode}")

# ── Anteprima ─────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(img_synth); ax1.set_title("Dataset A — Gradiente sintetico", fontweight="bold"); ax1.axis("off")
ax2.imshow(img_photo);  ax2.set_title("Dataset B — Immagine fotografica",  fontweight="bold"); ax2.axis("off")
plt.tight_layout()
plt.show()
print("✅ Dataset pronto.")

---
## Sezione 1 — Caricare e ispezionare un'immagine

### Concetti chiave

Un'immagine digitale è una **matrice 2D (o 3D) di pixel**. Pillow la rappresenta come oggetto `Image`, con attributi che descrivono la struttura del dato:

| Attributo | Significato | Esempio |
|-----------|-------------|---------|
| `.size` | Tupla `(larghezza, altezza)` in pixel | `(512, 512)` |
| `.mode` | Modalità di rappresentazione: `'L'` grigio, `'RGB'`, `'RGBA'`, `'P'` palette | `'RGB'` |
| `.format` | Formato del file sorgente (solo se aperto da file) | `'JPEG'` |

> ⚠️ **Attenzione:** `.size` restituisce `(larghezza, altezza)`, mentre NumPy usa la convenzione matriciale `(righe, colonne)` = `(altezza, larghezza)`. Questo invertimento è una fonte comune di bug.

In [ ]:
# ✅ Apertura e ispezione degli attributi principali
img = Image.open(OUTPUT_DIR / "photo.png")

print("=" * 45)
print(f"  size   : {img.size}   # (larghezza, altezza)")
print(f"  mode   : {img.mode}")
print(f"  format : {img.format}")
print("=" * 45)

# Dimensione dei soli campioni: qui ogni canale occupa 1 byte (uint8).
# Esclude metadati, intestazioni e memoria occupata dagli oggetti Python.
W, H    = img.size
canali  = len(img.getbands())
dim_kb  = W * H * canali / 1024
print(f"\nDimensione non compressa (teorica): {dim_kb:.1f} KiB")
print(f"  = {W} × {H} × {canali} byte")

# Visualizzazione
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.title(f"photo.png  |  {img.size}  |  mode={img.mode}")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Accesso ai pixel tramite array NumPy
arr = np.array(img)

print(f"Shape NumPy : {arr.shape}   # (altezza, larghezza, canali)")
print(f"dtype       : {arr.dtype}")
print(f"Range valori: [{arr.min()}, {arr.max()}]")
print()
print(f"Pixel alla riga=100, col=200 → {arr[100, 200]}")
print(f"  R={arr[100, 200, 0]}  G={arr[100, 200, 1]}  B={arr[100, 200, 2]}")

In [ ]:
# 🔧 ESERCIZIO 1.1
# Confrontate la dimensione teorica non compressa dell'immagine sintetica
# con quella effettiva del file PNG su disco.
# Calcolate il rapporto: byte dei pixel non compressi / byte del file PNG.

img_s = Image.open(OUTPUT_DIR / "synth.png")
file_size_kb = os.path.getsize(OUTPUT_DIR / "synth.png") / 1024

# --- Completate qui ---
# W, H = img_s.size
# canali = ...
# dim_teorica_kb = ...
# rapporto = ...
# print(f"Dimensione teorica : {dim_teorica_kb:.1f} KiB")
# print(f"Dimensione PNG     : {file_size_kb:.1f} KiB")
# print(f"Rapporto (lossless): {rapporto:.2f}x")

---
## Sezione 2 — Operazioni geometriche di base

Pillow offre un'API intuitiva per le trasformazioni più comuni. `resize()`, `crop()`, `rotate()` e `transpose()` restituiscono una **nuova immagine**. Altri metodi, come `thumbnail()`, modificano invece l’oggetto originale: lavorate su una `.copy()` se volete conservarlo.

### 2.1 — Ridimensionamento e filtri di ricampionamento

Quando cambiamo le dimensioni di un'immagine, ogni pixel di destinazione va calcolato a partire dai pixel di origine. Il **filtro di ricampionamento** stabilisce come:

| Filtro | Come calcola il nuovo pixel | Effetto tipico |
|--------|-----------------------------|----------------|
| `NEAREST` | Copia il valore del pixel di origine più vicino, senza interpolazione | Veloce; bordi a gradini e aliasing nel downsampling |
| `BILINEAR` | Media pesata lineare dei pixel vicini; in riduzione Pillow allarga il supporto in proporzione alla scala | Risultato più morbido, dettagli attenuati |
| `LANCZOS` | Sinc finestrata (finestra di Lanczos) su un supporto più ampio | Conserva meglio i dettagli; può introdurre *ringing* vicino ai bordi netti |

Nel grafico usiamo `interpolation="nearest"` in `imshow` per vedere i pixel effettivi, senza la levigatura aggiunta da matplotlib.

In [ ]:
# ✅ 2.1 — Ridimensionamento (resize) e filtri di ricampionamento

img = Image.open(OUTPUT_DIR / "photo.png")
TARGET = (128, 128)

filtri = {
    "NEAREST":  Image.Resampling.NEAREST,
    "BILINEAR": Image.Resampling.BILINEAR,
    "LANCZOS":  Image.Resampling.LANCZOS,
}

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(img)
axes[0].set_title(f"Originale\n{img.size}")
axes[0].axis("off")

for ax, (nome, filtro) in zip(axes[1:], filtri.items()):
    ridim = img.resize(TARGET, filtro)
    ax.imshow(ridim, interpolation="nearest")
    ax.set_title(f"{nome}\n{TARGET}")
    ax.axis("off")

plt.suptitle("Confronto filtri di ricampionamento — resize()", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()
print("Osservate le differenze ai bordi e nelle zone con dettaglio fine.")

### 2.1 (dettaglio) — Come si calcola un pixel dell'immagine ridimensionata

Per capire che cosa fanno i tre filtri, seguiamo il calcolo di **un singolo pixel** su due griglie piccole: un'immagine originale 4×4 in scala di grigio e la sua versione ingrandita a 6×6.

**Passo 1 — Riportare il pixel di destinazione sull'originale.** Pillow tratta ogni pixel come una cella quadrata. Il centro del pixel di destinazione nella colonna $c_{dst}$ e nella riga $r_{dst}$ viene riportato nelle coordinate dell'originale con

$$x_{src} = (c_{dst} + 0.5)\,s_x - 0.5, \qquad y_{src} = (r_{dst} + 0.5)\,s_y - 0.5, \qquad s_x = \frac{W_{src}}{W_{dst}},\; s_y = \frac{H_{src}}{H_{dst}}$$

In queste coordinate i valori interi corrispondono ai centri dei pixel originali. In generale $x_{src}$ e $y_{src}$ non sono interi: il punto cade *fra* i pixel originali.

**Passo 2 — Combinare i pixel vicini.** Qui i tre metodi si differenziano:

- **NEAREST** prende il valore del pixel originale che contiene il punto, cioè quello di indice $\lfloor (c_{dst}+0.5)\,s_x \rfloor$ (e analogamente per la riga). Nessuna media.
- **BILINEAR** usa i 4 pixel che circondano il punto. Posto $c_0 = \lfloor x_{src} \rfloor$, $r_0 = \lfloor y_{src} \rfloor$, $d_x = x_{src} - c_0$, $d_y = y_{src} - r_0$:

$$v = (1-d_x)(1-d_y)\,I[r_0, c_0] + d_x(1-d_y)\,I[r_0, c_0{+}1] + (1-d_x)\,d_y\,I[r_0{+}1, c_0] + d_x\,d_y\,I[r_0{+}1, c_0{+}1]$$

  I quattro pesi sono positivi e sommano a 1: il pixel più vicino al punto pesa di più.
- **LANCZOS** usa una finestra più ampia, fino a 3 pixel per lato in ogni direzione, con i pesi dati dal nucleo

$$L(t) = \operatorname{sinc}(t)\,\operatorname{sinc}(t/a) \;\text{ per } |t| < a, \qquad a = 3, \qquad \operatorname{sinc}(t) = \frac{\sin \pi t}{\pi t}$$

  Il filtro è **separabile**: il peso del pixel $(i, j)$ è $L(x_{src} - j)\cdot L(y_{src} - i)$, e Pillow lo applica in due passaggi, prima lungo le righe e poi lungo le colonne. Alcuni pesi sono **negativi**. I pixel che cadono fuori dall'immagine vengono esclusi e i pesi rimasti vengono normalizzati in modo che sommino a 1.

Nella figura, a sinistra c'è l'originale con i pixel usati da ciascun metodo (bordo arancione) e il loro peso $w$; il punto rosso indica $(x_{src}, y_{src})$. A destra c'è il risultato di `resize()`, con il pixel analizzato evidenziato.

In [ ]:
# ✅ 2.1a — Un esempio su griglie piccole: 4×4 → 6×6
# Immagine in scala di grigio con un bordo verticale netto fra la colonna 1 e la colonna 2.
src = np.array([
    [ 20,  40, 200, 220],
    [ 30,  60, 210, 240],
    [ 20,  50, 190, 230],
    [ 10,  40, 180, 220],
], dtype=np.uint8)
img_src = Image.fromarray(src)   # array 2D uint8 → modalità "L"

DST_SIZE = (6, 6)          # (larghezza, altezza) di destinazione
TARGET   = (2, 3)          # pixel di destinazione da analizzare: (riga, colonna)

W_src, H_src = img_src.size
W_dst, H_dst = DST_SIZE
sx, sy = W_src / W_dst, H_src / H_dst

# Centro del pixel di destinazione riportato nelle coordinate dell'immagine originale
r_dst, c_dst = TARGET
x_src = (c_dst + 0.5) * sx - 0.5
y_src = (r_dst + 0.5) * sy - 0.5
print(f"Fattore di scala   : sx = {W_src}/{W_dst} = {sx:.4f},  sy = {H_src}/{H_dst} = {sy:.4f}")
print(f"Pixel destinazione : riga {r_dst}, colonna {c_dst}")
print(f"Posizione sorgente : x = ({c_dst}+0.5)·{sx:.4f} − 0.5 = {x_src:.4f}")
print(f"                     y = ({r_dst}+0.5)·{sy:.4f} − 0.5 = {y_src:.4f}")

METODI = {
    "NEAREST":  Image.Resampling.NEAREST,
    "BILINEAR": Image.Resampling.BILINEAR,
    "LANCZOS":  Image.Resampling.LANCZOS,
}
risultati = {nome: np.array(img_src.resize(DST_SIZE, f)) for nome, f in METODI.items()}

def disegna_griglia(ax, arr, titolo, evidenzia=None, pesi=None, punto=None):
    """Disegna una griglia di pixel con il valore scritto in ogni cella."""
    ax.imshow(arr, cmap="gray", vmin=0, vmax=255)
    h, w = arr.shape
    for r in range(h):
        for c in range(w):
            colore = "white" if arr[r, c] < 128 else "black"
            testo = f"{arr[r, c]}"
            if pesi is not None and pesi[r, c] != 0:
                testo += f"\nw={pesi[r, c]:.3f}"
            ax.text(c, r, testo, ha="center", va="center", fontsize=8, color=colore)
    ax.set_xticks(range(w)); ax.set_yticks(range(h))
    ax.set_xticks(np.arange(-0.5, w), minor=True); ax.set_yticks(np.arange(-0.5, h), minor=True)
    ax.grid(which="minor", color="#94A3B8", lw=1)
    ax.tick_params(which="minor", length=0)
    ax.set_xlabel("colonna"); ax.set_ylabel("riga")
    if evidenzia is not None:
        for (r, c) in evidenzia:
            ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False, ec="#E07B39", lw=3))
    if punto is not None:
        ax.plot(*punto, "o", color="#E11D48", ms=9, mec="white", mew=1.5)
    ax.set_title(titolo, fontsize=10, fontweight="bold")

In [ ]:
# ✅ 2.1b — Calcolo del pixel di destinazione con i tre metodi

def tri(t):                      # nucleo del filtro bilineare (triangolo)
    t = abs(t)
    return 1 - t if t < 1 else 0.0

def lanczos(t, a=3):             # nucleo di Lanczos: sinc(t)·sinc(t/a) per |t| < a
    return float(np.sinc(t) * np.sinc(t / a)) if abs(t) < a else 0.0

def pesi_1d(centro, n, kernel, supporto):
    """Pesi lungo una dimensione per i pixel 0..n-1, normalizzati a somma 1.
    I pixel fuori dall'immagine vengono esclusi (come in Pillow)."""
    idx = np.arange(n)
    w = np.array([kernel(i - centro) if abs(i - centro) < supporto else 0.0 for i in idx])
    return w / w.sum()

# ── NEAREST ──────────────────────────────────────────────────────────────────
# Pillow sceglie il pixel di indice floor((c_dst + 0.5) · sx)
c_n = int(math.floor((c_dst + 0.5) * sx))
r_n = int(math.floor((r_dst + 0.5) * sy))
v_nearest = src[r_n, c_n]
print("NEAREST")
print(f"  pixel sorgente = (riga {r_n}, colonna {c_n})  →  valore {v_nearest}")
print(f"  Pillow: {risultati['NEAREST'][r_dst, c_dst]}\n")

# ── BILINEAR ─────────────────────────────────────────────────────────────────
c0, r0 = int(math.floor(x_src)), int(math.floor(y_src))
dx, dy = x_src - c0, y_src - r0
q00, q01 = int(src[r0, c0]),     int(src[r0, c0 + 1])
q10, q11 = int(src[r0 + 1, c0]), int(src[r0 + 1, c0 + 1])
v_bil = (q00 * (1 - dx) * (1 - dy) + q01 * dx * (1 - dy)
         + q10 * (1 - dx) * dy     + q11 * dx * dy)
print("BILINEAR")
print(f"  4 vicini: righe {r0}–{r0+1}, colonne {c0}–{c0+1};  dx = {dx:.4f}, dy = {dy:.4f}")
print(f"  v = {q00}·{(1-dx)*(1-dy):.4f} + {q01}·{dx*(1-dy):.4f} + {q10}·{(1-dx)*dy:.4f} + {q11}·{dx*dy:.4f}")
print(f"    = {v_bil:.2f}  →  arrotondato {round(v_bil)}")
print(f"  Pillow: {risultati['BILINEAR'][r_dst, c_dst]}\n")

# ── LANCZOS (a = 3), separabile: prima le righe, poi le colonne ──────────────
wx = pesi_1d(x_src, W_src, lanczos, 3)
wy = pesi_1d(y_src, H_src, lanczos, 3)
print("LANCZOS (a=3)")
print("  pesi orizzontali (colonne 0..3):", np.round(wx, 4))
print("  pesi verticali   (righe   0..3):", np.round(wy, 4))
# Pillow esegue due passaggi e arrotonda a 8 bit dopo il primo
orizz = np.clip(np.round(src.astype(float) @ wx), 0, 255)   # una colonna intermedia
v_lan = float(wy @ orizz)
print(f"  passaggio orizzontale (per ogni riga): {orizz.astype(int)}")
print(f"  passaggio verticale: v = {v_lan:.2f}  →  arrotondato {int(np.clip(round(v_lan), 0, 255))}")
print(f"  Pillow: {risultati['LANCZOS'][r_dst, c_dst]}")

# ── Figura: a sinistra la griglia 4×4 con i pixel coinvolti, a destra la 6×6 ─
pesi_bil = np.zeros((H_src, W_src))
pesi_bil[r0, c0], pesi_bil[r0, c0 + 1] = (1 - dx) * (1 - dy), dx * (1 - dy)
pesi_bil[r0 + 1, c0], pesi_bil[r0 + 1, c0 + 1] = (1 - dx) * dy, dx * dy
pesi_lan = np.outer(wy, wx)

casi = [
    ("NEAREST",  [(r_n, c_n)], None),
    ("BILINEAR", [(r0, c0), (r0, c0 + 1), (r0 + 1, c0), (r0 + 1, c0 + 1)], pesi_bil),
    ("LANCZOS",  [(r, c) for r in range(H_src) for c in range(W_src) if pesi_lan[r, c] != 0], pesi_lan),
]
fig, axes = plt.subplots(3, 2, figsize=(10, 14))
for (nome, celle, pesi), (ax_s, ax_d) in zip(casi, axes):
    disegna_griglia(ax_s, src, f"{nome} — originale {W_src}×{H_src}",
                    evidenzia=celle, pesi=pesi, punto=(x_src, y_src))
    disegna_griglia(ax_d, risultati[nome], f"{nome} — ridimensionata {W_dst}×{H_dst}",
                    evidenzia=[TARGET])
plt.suptitle("Il punto rosso è il centro del pixel di destinazione riportato sull'originale",
             fontsize=11, y=1.0)
plt.tight_layout()
plt.show()

### 🔍 Cose da osservare — interpolazione

1. **Il calcolo manuale coincide con Pillow** per il pixel analizzato con tutti e tre i metodi. Su altri pixel o altre immagini potete trovare differenze di ±1: Pillow lavora in virgola fissa e arrotonda a 8 bit fra il passaggio orizzontale e quello verticale. Cambiate `TARGET` e verificate.
2. **NEAREST duplica righe e colonne.** L'immagine 6×6 contiene solo valori già presenti nell'originale: il bordo resta netto, ma a gradini.
3. **BILINEAR resta nell'intervallo dei vicini.** Il valore calcolato è una media pesata con pesi positivi, quindi sta sempre fra il minimo e il massimo dei 4 pixel usati: non crea nuovi estremi e "spalma" il bordo su più colonne.
4. **LANCZOS può uscire dall'intervallo dell'originale.** Per via dei pesi negativi, accanto al bordo compaiono valori più scuri del lato scuro e più chiari del lato chiaro (nella colonna 1 della 6×6 i valori scendono sotto quelli della colonna 1 originale). È il *ringing*: rende i bordi più nitidi, ma può creare aloni.
5. **In riduzione il filtro si allarga.** Quando l'immagine viene rimpicciolita ($s > 1$), Pillow dilata il nucleo del fattore di scala: i pesi diventano $L\big((j - x_{src})/s\big)$ e la finestra si estende per $a \cdot s$ pixel per lato. Per il passaggio 512 → 128 della cella precedente ($s = 4$), BILINEAR media circa 8 pixel per direzione e LANCZOS circa 24. Il filtro agisce così da passa-basso e riduce l'aliasing. NEAREST non lo fa: per questo, nel confronto sulla fotografia, è il metodo con i bordi più seghettati.

### 2.2 — Ritaglio

`crop()` riceve una tupla `(left, upper, right, lower)` in pixel, con origine in alto a sinistra. `right` e `lower` sono esclusi: la dimensione del ritaglio è `(right − left, lower − upper)`.

Nel quadrante in alto a sinistra compare anche parte della banda bianca aggiunta in Sezione 0: il ritaglio opera sull'immagine 512×512, bande comprese.

In [ ]:
# ✅ 2.2 — Ritaglio (crop)

img = Image.open(OUTPUT_DIR / "photo.png")
W, H = img.size

box_topleft = (0, 0, W // 2, H // 2)                                       # quadrante alto-sinistra
box_center  = (int(W * 0.25), int(H * 0.25), int(W * 0.75), int(H * 0.75)) # centro 50%

crop_tl = img.crop(box_topleft)
crop_ct = img.crop(box_center)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (titolo, im) in zip(axes, [
    ("Originale", img),
    (f"crop{box_topleft}", crop_tl),
    (f"crop{box_center}",  crop_ct),
]):
    ax.imshow(im, interpolation="nearest"); ax.set_title(titolo, fontsize=9); ax.axis("off")

plt.suptitle("crop(left, upper, right, lower)", fontweight="bold")
plt.tight_layout()
plt.show()

### 2.3 — Rotazione e flip

`rotate(angolo)` ruota in senso **antiorario** di un angolo in gradi. Con `expand=False` il canvas resta della stessa dimensione e gli angoli vengono tagliati; con `expand=True` il canvas si allarga per contenere tutta l'immagine ruotata. Per default `rotate()` usa il filtro `NEAREST`: per angoli non multipli di 90° potete passare `resample=Image.Resampling.BICUBIC` per bordi meno seghettati.

`transpose()` esegue invece flip e rotazioni di multipli di 90° (anche `ROTATE_90` è antiorario): sono permutazioni esatte dei pixel, senza interpolazione.

In [ ]:
# ✅ 2.3 — Rotazione e flip

img = Image.open(OUTPUT_DIR / "photo.png")

varianti = [
    ("Originale",          img),
    ("rotate(45)",         img.rotate(45, expand=False, fillcolor=(230, 230, 230))),
    ("rotate(45, expand)", img.rotate(45, expand=True,  fillcolor=(230, 230, 230))),
    ("FLIP_LEFT_RIGHT",    img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)),
    ("FLIP_TOP_BOTTOM",    img.transpose(Image.Transpose.FLIP_TOP_BOTTOM)),
    ("ROTATE_90",          img.transpose(Image.Transpose.ROTATE_90)),
]

fig, axes = plt.subplots(1, 6, figsize=(18, 3.2))
for ax, (titolo, im) in zip(axes, varianti):
    ax.imshow(im, interpolation="nearest"); ax.set_title(titolo, fontsize=8.5); ax.axis("off")

plt.suptitle("rotate() e transpose() — trasformazioni geometriche", fontweight="bold")
plt.tight_layout()
plt.show()
print("expand=True ridimensiona il canvas per contenere l'immagine ruotata intera.")

### 2.4 — Mantenere le proporzioni

`resize((200, 200))` impone le dimensioni richieste e deforma un'immagine rettangolare. `thumbnail((200, 200))` riduce invece l'immagine finché entra nel riquadro, conservando le proporzioni. Partiamo dalla fotografia rettangolare originale (`photo_rect`, creata in Sezione 0), senza le bande bianche.

In [ ]:
# ✅ 2.4 — thumbnail() vs resize(): mantenere le proporzioni
img_rect = photo_rect.copy()
img_thumb = img_rect.copy()
img_thumb.thumbnail((200, 200), Image.Resampling.LANCZOS)  # modifica la copia
img_resized = img_rect.resize((200, 200), Image.Resampling.LANCZOS)
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, (title, im) in zip(axes, [
    ("Originale", img_rect), ("thumbnail: proporzioni conservate", img_thumb),
    ("resize: dimensioni imposte", img_resized),
]):
    ax.imshow(im, interpolation="nearest")
    ax.set_title(f"{title}\n{im.size}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("thumbnail modifica l'oggetto e restituisce None; resize restituisce una nuova immagine.")

In [ ]:
# 🔧 ESERCIZIO 2.1
# Partendo dall'immagine sintetica:
#   a) Create una nuova immagine 200×200 con resize e filtro LANCZOS
#   b) Ruotatela di 30° con expand=True e sfondo nero
#   c) Mostrate i due risultati affiancati
#
# Suggerimento: rotate(..., fillcolor="black") imposta lo sfondo.

img_s = Image.open(OUTPUT_DIR / "synth.png")

# --- Completate qui ---
# ridim   = ...
# ruotata = ...
#
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
# ax1.imshow(ridim);   ax1.set_title("resize 200×200"); ax1.axis("off")
# ax2.imshow(ruotata); ax2.set_title("Ruotata 30°");       ax2.axis("off")
# plt.tight_layout(); plt.show()

---
## Sezione 3 — Conversione tra modalità

La **modalità** (`mode`) definisce come Pillow interpreta i valori di ciascun pixel. La conversione si fa con `.convert(mode)`.

| Mode | Canali | Byte/pixel | Uso tipico |
|------|--------|-----------|------------|
| `'L'` | 1 (grigio) | 1 | Imaging medico, machine learning |
| `'RGB'` | 3 | 3 | Fotografie, display |
| `'RGBA'` | 4 (R,G,B,Alfa) | 4 | Web, compositing |
| `'P'` | 1 (indice palette) | 1 (più palette condivisa) | GIF, grafica piatta |
| `'CMYK'` | 4 | 4 | Stampa professionale |

### 3.1 — Da RGB a scala di grigio

`convert("L")` calcola per ogni pixel una media pesata dei tre canali secondo i coefficienti ITU-R BT.601:

$$L = 0.299\,R + 0.587\,G + 0.114\,B$$

I pesi diversi riflettono la diversa sensibilità dell'occhio ai tre primari: il verde contribuisce di più alla luminosità percepita. La formula si applica direttamente ai valori a 8 bit memorizzati, che sono codificati con gamma: il risultato è quindi una *luma*, non la luminanza fisica.

Nella figura i singoli canali sono mostrati con colormap rossa, verde e blu, con scala fissa 0–255: i toni chiari corrispondono a valori alti del canale.

In [ ]:
# ✅ 3.1 — RGB → Scala di grigio ('L')

img      = Image.open(OUTPUT_DIR / "photo.png")
img_gray = img.convert("L")

arr      = np.array(img)
arr_gray = np.array(img_gray)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for ax, (titolo, data, cm) in zip(axes, [
    ("Originale RGB",  arr,           None),
    ("Canale R",       arr[:,:,0],    "Reds_r"),
    ("Canale G",       arr[:,:,1],    "Greens_r"),
    ("Canale B",       arr[:,:,2],    "Blues_r"),
    ("Grigio 'L'",     arr_gray,      "gray"),
]):
    ax.imshow(data, cmap=cm, vmin=0, vmax=255, interpolation="nearest")
    ax.set_title(titolo, fontweight="bold")
    ax.axis("off")

plt.suptitle("Decomposizione RGB e conversione in scala di grigio", y=1.02)
plt.tight_layout()
plt.show()

print(f"RGB  : size={img.size}, mode={img.mode}, shape={arr.shape}")
print(f"Gray : size={img_gray.size}, mode={img_gray.mode}, shape={arr_gray.shape}")

### 3.2 — Canale alfa

`convert("RGBA")` aggiunge un quarto canale, l'**alfa**, che indica l'opacità: 255 = opaco, 0 = trasparente. La conversione da RGB imposta alfa = 255 ovunque. Per rendere visibile la trasparenza componiamo l'immagine su uno sfondo a scacchi:

$$C_{\text{out}} = \alpha\,C_{\text{img}} + (1-\alpha)\,C_{\text{sfondo}}, \qquad \alpha \in [0, 1]$$

In [ ]:
# ✅ 3.2 — RGB → RGBA: aggiunta del canale alfa (trasparenza)

img      = Image.open(OUTPUT_DIR / "photo.png")
img_rgba = img.convert("RGBA")

arr_rgba = np.array(img_rgba)
print(f"Shape RGBA: {arr_rgba.shape}   (H × W × 4)")
print(f"Canale alfa: min={arr_rgba[:,:,3].min()}, max={arr_rgba[:,:,3].max()}  (255 = opaco)")

# Creiamo un gradiente di trasparenza orizzontale
arr_fade = arr_rgba.copy()
W = arr_fade.shape[1]
arr_fade[:, :, 3] = np.linspace(255, 0, W, dtype=np.uint8)[np.newaxis, :]

img_fade = Image.fromarray(arr_fade)

# Compositing su sfondo a scacchi (pattern tipico per visualizzare trasparenza)
H_px, W_px = arr_fade.shape[:2]
tile = 32
xs, ys = np.arange(W_px) // tile, np.arange(H_px) // tile
cb = (((xs[np.newaxis, :] + ys[:, np.newaxis]) % 2) * 200 + 55).astype(np.uint8)
cb3 = np.stack([cb, cb, cb], axis=2)
alpha = arr_fade[:, :, 3:4] / 255.0
comp = (arr_fade[:, :, :3] * alpha + cb3 * (1 - alpha)).astype(np.uint8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.imshow(img);  ax1.set_title("Originale RGB");                  ax1.axis("off")
ax2.imshow(comp); ax2.set_title("RGBA con gradiente di trasparenza"); ax2.axis("off")
plt.suptitle("Canale alfa: 255=opaco, 0=trasparente", fontweight="bold")
plt.tight_layout()
plt.show()

### 3.3 — Approfondimento facoltativo: palette e quantizzazione

In modalità `'P'` (colore indicizzato) ogni pixel memorizza un **indice** in una tabella di al massimo 256 colori. È la rappresentazione usata, per esempio, da GIF e dai PNG a palette. Per passare da RGB a `'P'` serve una **quantizzazione cromatica**: `quantize()` sceglie una palette di N colori, qui con l'algoritmo *Median Cut*. L'algoritmo di quantizzazione non fa parte del formato: GIF specifica solo come memorizzare palette e indici.

In [ ]:
# ✅ APPROFONDIMENTO FACOLTATIVO — 3.3 — RGB → Palette ('P'): quantizzazione cromatica

img = Image.open(OUTPUT_DIR / "photo.png")

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
axes[0].imshow(img); axes[0].set_title("RGB (fino a 16.7M colori)", fontweight="bold"); axes[0].axis("off")

for ax, n_colori in zip(axes[1:], [2, 8, 32, 256]):
    img_q = img.quantize(colors=n_colori, method=Image.Quantize.MEDIANCUT).convert("RGB")
    ax.imshow(img_q)
    ax.set_title(f"{n_colori} colori")
    ax.axis("off")

plt.suptitle("Quantizzazione cromatica — modalità palette 'P'", fontweight="bold")
plt.tight_layout()
plt.show()
print("Con pochi colori emergono bordi netti tra zone (posterizzazione).")

> **Nota per l'esercizio 3.1.** La conversione diretta `convert("CMYK")` di Pillow non usa un profilo ICC: calcola C = 255 − R, M = 255 − G, Y = 255 − B e pone **K = 0** per ogni pixel, anche per il nero. Non è quindi una separazione pronta per la stampa, che richiede una gestione del colore con profili.

In [ ]:
# 🔧 ESERCIZIO 3.1 — FACOLTATIVO
# Convertite l'immagine sintetica in CMYK e visualizzate i 4 canali separatamente.
# Visualizzate i canali con vmin=0 e vmax=255; le colormap sono convenzionali.
# Verificate il minimo e il massimo di K, anche su un pixel RGB nero.

img_s = Image.open(OUTPUT_DIR / "synth.png")

# --- Completate qui ---
# img_cmyk = img_s.convert("CMYK")
# arr_cmyk = np.array(img_cmyk)
# nomi   = ["C (Cyan)", "M (Magenta)", "Y (Yellow)", "K (Key/Nero)"]
# cmaps  = ["Blues", "RdPu", "YlOrBr", "gray"]
# fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
# for ax, (nome, cmap) in zip(axes, zip(nomi, cmaps)):
#     ...

---
## Sezione 4 — Salvataggio in formati diversi e misurazione della dimensione

### Il quality factor JPEG

Il parametro `quality` (scala 0–95 secondo la documentazione di Pillow) controlla l'aggressività della quantizzazione dei coefficienti DCT mediante apposite tabelle:

- **quality bassa** → molti coefficienti azzerati → file piccolo, molti artefatti
- **quality alta** → poca perdita → file grande, artefatti quasi invisibili

Il valore di default di Pillow è **75** (buon compromesso per fotografie).
Valori oltre 95 sono sconsigliati: con `quality=100` Pillow disattiva parte della compressione e il file cresce molto, ma il JPEG resta lossy. Lo includiamo per osservarlo nelle misure. Usiamo `subsampling=0` (4:4:4) per mantenere costante il campionamento cromatico e studiare l’effetto di `quality`. Non esiste un ordine universale delle dimensioni BMP/PNG/JPEG: misuratelo sui due contenuti.

In [ ]:
# ✅ 4.1 — Salvataggio in BMP, PNG (vari livelli), JPEG (vari quality)

img = Image.open(OUTPUT_DIR / "photo.png")

# BMP: nessuna compressione
img.save(OUTPUT_DIR / "out.bmp")

# PNG: compress_level cambia tempo e dimensione, ma non i pixel (lossless)
for lvl in [0, 6, 9]:
    img.save(OUTPUT_DIR / f"out_png_c{lvl}.png", compress_level=lvl)

# JPEG: quality regola il trade-off qualità/dimensione
QUALITY_VALUES = [5, 10, 20, 35, 50, 65, 75, 85, 95, 100]
for q in QUALITY_VALUES:
    img.save(OUTPUT_DIR / f"out_q{q:03d}.jpg", quality=q, subsampling=0)

# Riepilogo dimensioni
print(f"{'File':<26} {'Dimensione':>12}")
print("-" * 40)
for nome in ["out.bmp",
              "out_png_c0.png", "out_png_c6.png", "out_png_c9.png"]:
    sz = os.path.getsize(OUTPUT_DIR / nome) / 1024
    print(f"{nome:<26} {sz:>10.1f} KiB")
print()
for q in QUALITY_VALUES:
    nome = f"out_q{q:03d}.jpg"
    sz = os.path.getsize(OUTPUT_DIR / nome) / 1024
    print(f"{nome:<26} {sz:>10.1f} KiB")
print("\n✅ Tutti i file salvati.")

In [ ]:
# ✅ 4.2 — Curva dimensione file vs quality factor

sizes_jpg = {q: os.path.getsize(OUTPUT_DIR / f"out_q{q:03d}.jpg") / 1024
             for q in QUALITY_VALUES}

sz_bmp    = os.path.getsize(OUTPUT_DIR / "out.bmp")         / 1024
sz_png_c6 = os.path.getsize(OUTPUT_DIR / "out_png_c6.png")  / 1024

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Grafico 1: curva JPEG + linee di riferimento ──────────────────────────────
ax = axes[0]
ax.plot(list(sizes_jpg.keys()), list(sizes_jpg.values()),
        "-o", color="#E07B39", linewidth=2, markersize=5, label="JPEG")
ax.axhline(sz_bmp,    color="#64748B", ls="--", lw=1.5, label=f"BMP ({sz_bmp:.0f} KiB)")
ax.axhline(sz_png_c6, color="#0D7377", ls="--", lw=1.5, label=f"PNG c=6 ({sz_png_c6:.0f} KiB)")
ax.set_xlabel("Quality factor JPEG",  fontsize=12)
ax.set_ylabel("Dimensione file (KiB)", fontsize=12)
ax.set_title("Dimensione vs Quality factor", fontweight="bold")
ax.legend(); ax.grid(True, alpha=0.3)

# ── Grafico 2: confronto a barre ──────────────────────────────────────────────
ax2 = axes[1]
etichette  = ["BMP", "PNG c=0", "PNG c=6", "PNG c=9",
               "JPEG q=50", "JPEG q=75", "JPEG q=95"]
dimensioni = [
    sz_bmp,
    os.path.getsize(OUTPUT_DIR / "out_png_c0.png") / 1024,
    sz_png_c6,
    os.path.getsize(OUTPUT_DIR / "out_png_c9.png") / 1024,
    sizes_jpg[50], sizes_jpg[75], sizes_jpg[95],
]
colori = ["#64748B", "#0D9488", "#0D7377", "#065A82",
          "#F59E0B", "#E07B39", "#C0392B"]
bars = ax2.bar(etichette, dimensioni, color=colori, width=0.6)
for bar, dim in zip(bars, dimensioni):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
             f"{dim:.0f} KiB", ha="center", va="bottom", fontsize=9)
ax2.set_ylabel("Dimensione file (KiB)", fontsize=12)
ax2.set_title("Confronto formati — stesso contenuto", fontweight="bold")
ax2.set_ylim(0, max(dimensioni) * 1.2)
ax2.grid(True, alpha=0.3, axis="y")
plt.xticks(rotation=20, ha="right")

plt.tight_layout()
plt.show()

print(f"Rapporto BMP / JPEG(q=75): {sz_bmp / sizes_jpg[75]:.1f}x")
print(f"Rapporto PNG / JPEG(q=75): {sz_png_c6 / sizes_jpg[75]:.1f}x")

### 🔍 Cose da osservare — formati e dimensioni

I valori citati vengono da un'esecuzione con Pillow 12.3 sulla fotografia 512×512: confrontateli con i vostri, che possono differire leggermente con versioni diverse di Pillow e zlib.

1. **BMP e PNG `compress_level=0`.** Il PNG non compresso è leggermente *più grande* del BMP (nella nostra esecuzione 768,8 contro 768,1 KiB). Perché? Pensate a cosa aggiunge PNG: un byte di filtro per riga, blocchi zlib "stored", intestazioni dei chunk e CRC.
2. **PNG `compress_level=6` contro `9`.** Il livello 9 riduce il file di pochi punti percentuali (circa il 4% qui) e richiede più tempo. I pixel decodificati sono identici: cambia solo lo sforzo del compressore.
3. **Forma della curva JPEG.** Fra q=5 e q=85 la dimensione cresce lentamente (da circa 9 a circa 30 KiB); sopra 85 la curva si impenna. Fra q=95 e q=100 il file diventa circa 2,5 volte più grande. Dove conviene posizionarsi per una fotografia?
4. **Ordine dei formati.** Su questa fotografia JPEG q=75 è decine di volte più piccolo di BMP e diverse volte più piccolo di PNG. Prima di eseguire l'esercizio 4.1, provate a prevedere: l'ordine sarà lo stesso sul gradiente sintetico?
5. **Bande bianche.** Circa un terzo dell'immagine è costituito dalle bande uniformi aggiunte in Sezione 0. Come pensate che influiscano sulle dimensioni di PNG e JPEG?

In [ ]:
# 🔧 ESERCIZIO 4.1
# Ripetete la stessa analisi usando l'immagine SINTETICA.
# Come cambia la curva dimensione/quality rispetto alla fotografica?
# Perché il PNG comprime il gradiente diversamente da una fotografia?
# (Suggerimento: pensate all'entropia del contenuto.)

img_s = Image.open(OUTPUT_DIR / "synth.png")

# --- Completate qui ---

---
## Sezione 5 — Visualizzazione comparativa e artefatti JPEG

### Artefatti della compressione JPEG

La DCT opera su **blocchi 8×8 pixel**. La quantizzazione aggressiva di questi blocchi introduce artefatti caratteristici:

| Artefatto | Causa | Dove si vede |
|-----------|-------|--------------|
| **Blocchettizzazione** | Confini visibili tra blocchi DCT 8×8 | quality < 20, zone uniformi |
| **Ringing** | Troncamento frequenze alte → oscillazioni di Gibbs | bordi netti, testi |
| **Perdita di dettaglio** | Coefficienti ad alta frequenza → 0 | texture fini, pelo, foglie |
| **Color bleeding** | Crominanza sub-campionata (4:2:0) | bordi tra colori saturi |

Negli esempi seguenti usiamo 4:4:4: non mostriamo quindi la perdita dovuta al sottocampionamento 4:2:0. Il confronto specifico è nell’esercizio 6.2. I valori di quality nella tabella sono indicativi, non soglie universali.

In [ ]:
# ✅ 5.1 — Confronto visivo: BMP, PNG e JPEG a quality diverse

formati_confronto = [
    ("BMP (no compressione)",  "out.bmp"),
    ("PNG lossless",           "out_png_c6.png"),
    ("JPEG q=95",              "out_q095.jpg"),
    ("JPEG q=50",              "out_q050.jpg"),
    ("JPEG q=20",              "out_q020.jpg"),
    ("JPEG q=5",               "out_q005.jpg"),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 9))
for ax, (titolo, nome_file) in zip(axes.flat, formati_confronto):
    im = Image.open(OUTPUT_DIR / nome_file)
    sz = os.path.getsize(OUTPUT_DIR / nome_file) / 1024
    ax.imshow(im, interpolation="nearest")
    ax.set_title(f"{titolo}\n{sz:.1f} KiB", fontweight="bold", fontsize=10)
    ax.axis("off")

plt.suptitle("Confronto formati — stessa immagine, compressioni diverse",
             fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ✅ 5.2 — Zoom sugli artefatti: analisi locale
# Ritagliamo la stessa zona da più versioni e amplifichiamo l'errore.

ZOOM_BOX = (160, 160, 320, 320)   # (left, upper, right, lower)
ZOOM_SIZE = (384, 384)             # dimensione di visualizzazione

ref    = Image.open(OUTPUT_DIR / "out_png_c6.png")
j_95   = Image.open(OUTPUT_DIR / "out_q095.jpg")
j_5    = Image.open(OUTPUT_DIR / "out_q005.jpg")

z_ref  = ref.crop(ZOOM_BOX).resize(ZOOM_SIZE, Image.Resampling.NEAREST)
z_95   = j_95.crop(ZOOM_BOX).resize(ZOOM_SIZE, Image.Resampling.NEAREST)
z_5    = j_5.crop(ZOOM_BOX).resize(ZOOM_SIZE,  Image.Resampling.NEAREST)

def errore_amplificato(a, b, fattore=5):
    diff = np.abs(np.array(a).astype(int) - np.array(b).astype(int)) * fattore
    return Image.fromarray(np.clip(diff, 0, 255).astype(np.uint8))

err_95 = errore_amplificato(z_ref, z_95)
err_5  = errore_amplificato(z_ref, z_5)

fig, axes = plt.subplots(2, 3, figsize=(13, 9))
for ax, (titolo, im) in zip(axes[0], [
    ("Riferimento (PNG)", z_ref),
    ("JPEG q=95",         z_95),
    ("JPEG q=5",          z_5),
]):
    ax.imshow(im, interpolation="nearest"); ax.set_title(titolo, fontweight="bold"); ax.axis("off")

axes[1, 0].axis("off")
for ax, (titolo, im) in zip(axes[1, 1:], [
    ("Errore ×5 (q=95)", err_95),
    ("Errore ×5 (q=5)",  err_5),
]):
    ax.imshow(im, interpolation="nearest"); ax.set_title(titolo, fontweight="bold"); ax.axis("off")

plt.suptitle(f"Zoom zona {ZOOM_BOX} — artefatti JPEG visibili", fontweight="bold")
plt.tight_layout()
plt.show()

### 🔍 Cose da osservare — artefatti

1. **q=95.** A occhio l'immagine è indistinguibile dal riferimento, ma l'errore amplificato ×5 non è nullo: JPEG resta lossy anche a qualità alta.
2. **q=5 — blocchi.** Nello zoom si vedono blocchi quadrati regolari. Misurateli: corrispondono ai blocchi DCT 8×8 (nello zoom ogni pixel è ingrandito 2,4 volte).
3. **q=5 — colore.** Compaiono blocchi con tinte assenti nell'originale (per esempio verdastre). Anche senza sottocampionamento cromatico, la quantizzazione grossolana dei coefficienti di crominanza altera i colori.
4. **Dove si concentra l'errore.** Nell'errore amplificato a q=5 i valori più alti stanno lungo le zampe e le antenne, cioè vicino ai bordi netti, mentre lo sfondo sfocato cambia poco. Collegatelo alla tabella degli artefatti: quali sono legati alle alte frequenze?

### 5.3 — Approfondimento facoltativo: MSE e PSNR

Per misurare numericamente la differenza fra riferimento e immagine compressa usiamo:

$$\text{MSE} = \frac{1}{N}\sum_{i=1}^{N} (x_i - y_i)^2 \qquad \text{PSNR} = 10 \log_{10}\frac{255^2}{\text{MSE}} \ \text{dB}$$

dove la media è calcolata su tutti i campioni (pixel × canali) e 255 è il valore massimo per dati a 8 bit. Se MSE = 0 il PSNR è infinito. Un PSNR più alto indica meno errore numerico, ma non garantisce una migliore qualità percepita.

In [ ]:
# ✅ APPROFONDIMENTO FACOLTATIVO — 5.3: PSNR e dimensione file

def psnr(ref_img, comp_img):
    """Peak Signal-to-Noise Ratio (dB). Più alto = meno distorsione."""
    a = np.array(ref_img).astype(float)
    b = np.array(comp_img).astype(float)
    mse = ((a - b) ** 2).mean()
    return 10 * math.log10(255**2 / mse) if mse > 0 else float("inf")

ref = Image.open(OUTPUT_DIR / "out_png_c6.png")

for q in [5, 95]:
    print(f"JPEG q={q:<3} →  PSNR={psnr(ref, Image.open(OUTPUT_DIR / f'out_q{q:03d}.jpg')):.2f} dB")

psnr_vals = {q: psnr(ref, Image.open(OUTPUT_DIR / f"out_q{q:03d}.jpg"))
             for q in QUALITY_VALUES}
size_vals = {q: os.path.getsize(OUTPUT_DIR / f"out_q{q:03d}.jpg") / 1024
             for q in QUALITY_VALUES}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# PSNR vs quality
ax1.plot(list(psnr_vals.keys()), list(psnr_vals.values()),
         "-o", color="#0D7377", linewidth=2, markersize=5)
ax1.axhline(30, color="#E07B39", ls=":", label="30 dB — riferimento numerico")
ax1.axhline(40, color="#2D9E6B", ls=":", label="40 dB — riferimento numerico")
ax1.set_xlabel("Quality factor"); ax1.set_ylabel("PSNR (dB)")
ax1.set_title("PSNR vs Quality factor", fontweight="bold")
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

# Curva rate-distortion
ax2.plot(list(size_vals.values()), list(psnr_vals.values()),
         "-o", color="#C0392B", linewidth=2, markersize=5)
for q in [5, 20, 50, 75, 95]:
    ax2.annotate(f"q={q}",
                 xy=(size_vals[q], psnr_vals[q]),
                 xytext=(size_vals[q] + 4, psnr_vals[q] - 0.8),
                 fontsize=8, color="#64748B")
ax2.set_xlabel("Dimensione file (KiB)"); ax2.set_ylabel("PSNR (dB)")
ax2.set_title("Curva Rate-Distortion", fontweight="bold")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Confrontate q=75, 95 e 100: quanto crescono PSNR e dimensione? Il risultato dipende dal contenuto; le linee a 30 e 40 dB non sono soglie percettive.")

### 🔍 Cose da osservare — PSNR

1. **Andamento.** Nella nostra esecuzione il PSNR passa da circa 27 dB (q=5) a circa 40 dB (q=75) e 48 dB (q=95). La crescita non è lineare nel parametro `quality`.
2. **Rendimenti decrescenti.** Da q=75 a q=95 il PSNR guadagna circa 8 dB, ma la dimensione più che raddoppia. Da q=95 a q=100 il guadagno è di circa 6,6 dB a fronte di un file 2,5 volte più grande.
3. **PSNR e percezione.** Confrontate i valori con le immagini della Sezione 5.1: a quale PSNR smettete di notare differenze? Le linee a 30 e 40 dB sono solo riferimenti numerici, non soglie percettive.
4. **Dove nasce l'errore.** Il PSNR è una media su tutta l'immagine, comprese le bande bianche, dove l'errore è più basso. Un PSNR calcolato solo sulla zona della formica sarebbe più alto o più basso?

---
## Sezione 6 — Esercizi ed esperimenti
Per la prima lezione scegliete un confronto guidato; gli altri esercizi sono facoltativi. Prima delle attività sul PSNR eseguite l’approfondimento 5.3.


In [ ]:
# 🔧 ESERCIZIO 6.1 — PSNR sull'immagine sintetica
#
# Ripetete l'analisi PSNR della Sezione 5.3 usando img_synth.
# Confrontate le due curve rate-distortion sullo stesso grafico.
# Quale immagine tollera meglio la compressione JPEG? Perché?

# --- Completate qui ---

In [ ]:
# 🔧 ESERCIZIO 6.2 — Subsampling cromatico
#
# Salvate la stessa immagine a q=75 con:
#   - subsampling=0  (4:4:4, nessuna riduzione della crominanza)
#   - subsampling=2  (4:2:0, metà larghezza e metà altezza: 1/4 dei campioni per canale cromatico)
# Confrontate dimensione su disco e PSNR.
# In quale zona dell'immagine vedete più differenza?

img = Image.open(OUTPUT_DIR / "photo.png")

# --- Completate qui ---
# img.save(OUTPUT_DIR / "out_444.jpg", quality=75, subsampling=0)
# img.save(OUTPUT_DIR / "out_420.jpg", quality=75, subsampling=2)
# ...

> **Nota per l'esercizio 6.3.** Il JPEG *baseline* codifica i blocchi in un'unica scansione, dall'alto in basso. Il JPEG **progressivo** distribuisce i coefficienti DCT in più scansioni successive, che aggiungono coefficienti e/o ne raffinano la precisione. Le dimensioni in pixel restano uguali; un visualizzatore compatibile può mostrare anteprime via via più nitide durante lo scaricamento.

In [ ]:
# 🔧 ESERCIZIO 6.3 — JPEG progressivo
#
# Salvate la stessa immagine come JPEG standard e JPEG progressivo.
#
# img.save(OUTPUT_DIR / "out_standard.jpg",    quality=75, progressive=False)
# img.save(OUTPUT_DIR / "out_progressive.jpg", quality=75, progressive=True)
#
# Confrontate le dimensioni. Quale è più piccolo? La differenza dipende
# dal contenuto (fotografia vs grafica)?

# --- Completate qui ---

In [ ]:
# 🔧 ESERCIZIO 6.4 — Degradazione cumulativa (JPEG generazionale)
#
# Cosa succede se si apre e risalva un JPEG più volte?
# La ricompressione può introdurre perdita aggiuntiva, ma a parametri fissi
# possono comparire stabilizzazioni: il peggioramento non è sempre monotono.
# Simulate 10 generazioni a quality=50 e calcolate il PSNR a ogni passo.

GENERAZIONI = 10
QUALITY_GEN = 50

# --- Completate qui ---
# from io import BytesIO
# img_iter = Image.open(OUTPUT_DIR / "out_q075.jpg")
# ref_arr  = np.array(img_iter).astype(float)  # riferimento: JPEG iniziale, non originale
# psnr_gen = []
#
# for i in range(GENERAZIONI):
#     buf = BytesIO()
#     img_iter.save(buf, format="JPEG", quality=QUALITY_GEN, subsampling=0)
#     buf.seek(0)
#     img_iter = Image.open(buf).copy()
#     mse = ((ref_arr - np.array(img_iter).astype(float)) ** 2).mean()
#     psnr_gen.append(10 * math.log10(255**2 / mse) if mse > 0 else float("inf"))
#
# plt.plot(range(1, GENERAZIONI+1), psnr_gen, "-o", color="#C0392B")
# plt.xlabel("Generazione"); plt.ylabel("PSNR (dB)")
# plt.title(f"Degradazione JPEG cumulativa (q={QUALITY_GEN})")
# plt.grid(True, alpha=0.3); plt.show()

---
## Riepilogo

<div style="background:#F0FDF9; border-left:5px solid #0D7377; padding:20px 24px; border-radius:6px">

### Cosa abbiamo imparato

1. **`Image.open()` e attributi** — accesso immediato a `size`, `mode`, `format`; conversione in NumPy con `np.array()`. Ricordate che `size` è `(W, H)` mentre NumPy usa `(H, W, C)`.

2. **Trasformazioni geometriche** — `resize()`, `crop()`, `rotate()`, `transpose()` restituiscono sempre una nuova immagine. `thumbnail()` modifica invece l’oggetto preservando le proporzioni; LANCZOS è utile per il downsampling, ma può introdurre ringing.

3. **Conversione di modalità** — `convert('L')` usa i pesi ITU-R (G conta di più); `convert('RGBA')` aggiunge alfa=255; `quantize()` riduce i colori con Median Cut.

4. **Formato e dimensione** — Le dimensioni dipendono da contenuto e parametri. PNG preserva i pixel forniti al salvataggio (`compress_level` cambia tempo e dimensione). JPEG è lossy con trade-off regolabile.

5. **Artefatti JPEG** — la quantizzazione può produrre blocchettizzazione, ringing e perdita di dettaglio; il sottocampionamento cromatico può causare color bleeding. Il PSNR misura l’errore numerico, non direttamente la qualità percepita. Il compromesso fra dimensione e qualità va osservato sul contenuto specifico.

</div>

---

### Riferimenti

- Ze-Nian Li, Mark S. Drew, Jiangchuan Liu — *Fundamentals of Multimedia*, Springer — cap. 3 (rappresentazione delle immagini e formati) e cap. 9 (JPEG)
- [Pillow documentation](https://pillow.readthedocs.io/en/stable/) — API completa
- [matplotlib documentation](https://matplotlib.org/stable/) — visualizzazione
- Lezioni teoriche del corso: **Lezione 2** (Rappresentazione digitale delle immagini), **Lezione 3** (Formati popolari per immagini)